In [0]:
# Workspace paths (read/write for files, but Delta writes require Unity Catalog)
landing_zone_path = "/Workspace/flight_project/landing_zone/"
bronze_path = "/Workspace/flight_project/bronze/"
silver_path = "/Workspace/flight_project/silver/"
gold_path = "/Workspace/flight_project/gold/"

# Create the directories
dbutils.fs.mkdirs(landing_zone_path)
dbutils.fs.mkdirs(bronze_path)
dbutils.fs.mkdirs(silver_path)
dbutils.fs.mkdirs(gold_path)

print("✅ Workspace directories created successfully!")

In [0]:
dbutils.fs.ls("/Workspace/flight_project/")



In [0]:
dbutils.fs.cp("/Workspace/Users/tamilvanankannappan@gmail.com/Flight data ETL/airlines_flights_data.csv", "/Workspace/flight_project/landing_zone/")

In [0]:
dbutils.fs.ls("/Workspace/flight_project/landing_zone")

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Create schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS deltalaketamil.flight_project")

# Define the checkpoint path (Crucial for streaming!)
checkpoint_path = "/Workspace/flight_project/checkpoints/bronze/"

print("Starting Auto Loader ingestion stream...")

# 1. READ STREAM: Configure Auto Loader to watch the landing zone
bronze_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path) # Where it stores the inferred schema
    .option("header", "true")
    .load(landing_zone_path)
)

# 2. ENRICH: Add audit columns (Best Practice)
bronze_enriched_df = (bronze_df
    .withColumn("ingest_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

# 3. WRITE STREAM: Write to Unity Catalog managed table
bronze_query = (bronze_enriched_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path) # Where it tracks processed files
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("deltalaketamil.flight_project.bronze_flights")
)

# Wait for the micro-batch to finish processing
bronze_query.awaitTermination()

print("✅ Ingestion complete! Data written to Bronze layer.")

In [0]:
%sql
SELECT * FROM deltalaketamil.flight_project.bronze_flights LIMIT 5


In [0]:
from pyspark.sql.functions import col, when, current_timestamp, trim

print("Starting Silver layer transformation...")

# 1. READ: Load the raw data from your Bronze table
bronze_df = spark.table("deltalaketamil.flight_project.bronze_flights")

# 2. TRANSFORM: Cleanse, cast, and standardize
silver_df = (bronze_df
    # Drop pandas/CSV artifacts and Auto Loader rescue columns that we don't need in Silver
    .drop("index", "_rescued_data")
    
    # Drop exact duplicates just in case the source system sent identical records
    .dropDuplicates()
    
    # Trim accidental whitespace from string columns (very common in CSVs)
    .withColumn("airline", trim(col("airline")))
    .withColumn("source_city", trim(col("source_city")))
    .withColumn("destination_city", trim(col("destination_city")))
    
    # Cast numerical values to proper data types
    .withColumn("duration", col("duration").cast("double"))
    .withColumn("days_left", col("days_left").cast("integer"))
    .withColumn("price", col("price").cast("integer"))
    
    # Standardize the 'stops' column from text to integer using conditionally logic
    .withColumn("stops", 
        when(col("stops") == "zero", 0)
        .when(col("stops") == "one", 1)
        .when(col("stops") == "two_or_more", 2)
        .otherwise(None) # Handles any unexpected values gracefully
    )
    
    # Add a Silver layer audit timestamp
    .withColumn("silver_timestamp", current_timestamp())
)

# 3. WRITE: Save to the Silver layer in Unity Catalog
(silver_df.write
    .format("delta")
    .mode("overwrite") 
    .saveAsTable("deltalaketamil.flight_project.silver_flights")
)

print("✅ Data cleansed, standardized, and written to the Silver layer!")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import avg, count, round, current_timestamp

print("Starting Gold layer aggregation and upsert...")

# 1. READ: Load the cleansed Silver data
silver_df = spark.table("deltalaketamil.flight_project.silver_flights")

# 2. TRANSFORM: Create the business-level aggregation
gold_updates_df = (silver_df
    .groupBy("airline", "source_city", "destination_city", "class")
    .agg(
        round(avg("price"), 2).alias("avg_price"),
        round(avg("duration"), 2).alias("avg_duration"),
        count("flight").alias("total_flights")
    )
    .withColumn("gold_updated_at", current_timestamp())
)

# 3. WRITE / MERGE: Upsert into the Gold table
target_table_name = "deltalaketamil.flight_project.gold_route_stats"

# Check if the table exists to decide between an initial CREATE or an incremental MERGE
if not spark.catalog.tableExists(target_table_name):
    print("Gold table does not exist. Creating it for the first time...")
    gold_updates_df.write.format("delta").saveAsTable(target_table_name)
else:
    print("Gold table exists. Performing MERGE (Upsert)...")
    target_table = DeltaTable.forName(spark, target_table_name)
    
    # The MERGE logic matches on the unique combination of the route details
    (target_table.alias("target")
        .merge(
            gold_updates_df.alias("source"),
            """
            target.airline = source.airline AND 
            target.source_city = source.source_city AND 
            target.destination_city = source.destination_city AND 
            target.class = source.class
            """
        )
        .whenMatchedUpdateAll()     # If the route exists, update the averages
        .whenNotMatchedInsertAll()  # If it's a brand new route, insert it
        .execute()
    )

# 4. OPTIMIZE & Z-ORDER: Performance tuning for downstream queries
print("Optimizing Gold table and applying Z-Order...")
spark.sql(f"OPTIMIZE {target_table_name} ZORDER BY (source_city, destination_city)")

print("✅ Gold layer is fully updated, optimized, and ready for BI Analytics!")